[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

*Authors: Tutorial Series on Digital Image Processing*

## **Image Filtering: Concepts, Techniques, and Applications**


## Part 1: Introduction

Images captured by cameras or medical scanners are rarely perfect. They contain noise, unwanted artefacts, or fine details that need to be either suppressed or enhanced before further analysis. **Image filtering** is the set of mathematical operations that lets us accomplish exactly that — selectively modifying pixel values in a principled, reproducible way.

By the end of this notebook you will be able to:
- Understand what a convolution kernel is and how it transforms an image.
- Apply and compare a variety of classical filters (blur, sharpen, edge-detect, median, bilateral, Gabor).
- Reason about *when* each filter is appropriate and what trade-offs it involves.
- Use filters as pre-processing steps in a practical image analysis workflow.

### Preparing the notebook

Let's begin by installing and importing every library we'll need.


In [ ]:
# Install any libraries not already present in the environment
!pip install scikit-image opencv-python-headless matplotlib numpy scipy PyWavelets -q


In [ ]:
# Standard imports
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
import cv2
from scipy import ndimage
from skimage import data, filters, feature, restoration, util, exposure
from skimage.io import imread
from skimage.color import rgb2gray
from skimage.util import img_as_float, img_as_ubyte, random_noise
from skimage.filters import gaussian as sk_gauss, sobel, sobel_h, sobel_v, gabor
from skimage.feature import canny
from skimage.restoration import denoise_nl_means, estimate_sigma
import os, random

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
print("All libraries imported successfully!")


### Helper utilities

Throughout this notebook we will repeatedly want to display an original image side-by-side with a filtered version.


In [ ]:
def show_images(images, titles, cmap="gray", figsize=(16,4), vmin=None, vmax=None):
    """Display a list of images with titles in a single row."""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1: axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(title, fontsize=12); ax.axis("off")
    plt.tight_layout(); plt.show()

def show_kernels(kernels, titles, figsize=(14,3)):
    """Visualise small convolution kernels as annotated heatmaps."""
    n = len(kernels)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1: axes = [axes]
    for ax, k, title in zip(axes, kernels, titles):
        lim = max(np.abs(k).max(), 1e-6)
        ax.imshow(k, cmap="RdBu_r", vmin=-lim, vmax=lim)
        ax.set_title(title, fontsize=11)
        for (i,j), v in np.ndenumerate(k):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    fontsize=8, color="black")
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()

print("Helper functions defined.")


---

## Part 2: The Convolution Operation — The Engine Behind Every Filter

### What is convolution?

Every filter covered in this notebook works by *sliding a small matrix — called a **kernel** or **filter mask** — across the image* and computing a weighted sum of pixel values at every position.

Mathematically, for a 2-D image $I$ and kernel $K$ of size $(2r+1)\times(2r+1)$:

$$
(I * K)[x, y] = \sum_{i=-r}^{r} \sum_{j=-r}^{r} I[x-i,\, y-j] \cdot K[i,j]
$$

The step-by-step illustration below shows how a single output pixel is computed:


<sub><b>Figure:</b> Left: Cameraman — a classic signal-processing benchmark. Right: Retina scan — representative of medical imaging.</sub>


In [ ]:
# Load images
camera = img_as_float(data.camera())          # 512×512 grayscale
retina = img_as_float(rgb2gray(data.retina())) # 1411×1411 → greyscale

print(f"Camera : shape={camera.shape}, dtype={camera.dtype}")
print(f"Retina : shape={retina.shape}, dtype={retina.dtype}")
show_images([camera, retina],
            ["Cameraman (natural image)", "Retina scan (medical image)"], figsize=(10,4))


In [ ]:
# Kernel definitions used throughout this notebook
box_3   = np.ones((3,3)) / 9.0
gaussian_5 = np.array([[1,4,7,4,1],[4,16,26,16,4],[7,26,41,26,7],
                        [4,16,26,16,4],[1,4,7,4,1]], dtype=float)
gaussian_5 /= gaussian_5.sum()
sharpen  = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=float)
sobel_xk = np.array([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=float)
sobel_yk = np.array([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=float)
laplace  = np.array([[0,1,0],[1,-4,1],[0,1,0]], dtype=float)

show_kernels([box_3, gaussian_5, sharpen, sobel_xk, sobel_yk, laplace],
             ["Box blur", "Gaussian (5×5)", "Sharpen", "Sobel-X", "Sobel-Y", "Laplacian"])


---

## Part 3: Smoothing / Blurring Filters

Smoothing filters reduce high-frequency content (fine detail and noise) at the cost of some sharpness. They are almost always the *first* step in a processing pipeline.

### 3.1 Box (Mean) Filter

Replace each pixel with the *arithmetic mean* of its neighbourhood. A $k\times k$ box filter has $k^2$ equal weights of $1/k^2$.

**Pros:** Extremely fast.  **Cons:** Treats all neighbours equally; creates artefacts on edges.

<sub><b>Figure:</b> Box filter at three kernel sizes — larger kernels produce stronger (and less realistic) blurring.</sub>


In [ ]:
box_results = [cv2.blur(camera, (k,k)) for k in [3, 9, 21]]
show_images([camera] + box_results,
            ["Original", "Box 3×3", "Box 9×9", "Box 21×21"], figsize=(18,4))


---

> **Note**: Notice how the edges of the cameraman's silhouette become progressively smeared as the kernel grows. This trade-off between smoothing strength and edge preservation is central to every filter choice.

---

### 3.2 Gaussian Filter

Weights neighbours by a 2-D Gaussian bell curve. Pixels close to the centre get high weight; distant ones get very little. Spread is controlled by standard deviation $\sigma$.

$$G(x,y) = \frac{1}{2\pi\sigma^2}\, e^{-\frac{x^2+y^2}{2\sigma^2}}$$

<sub><b>Figure:</b> Gaussian filter at three σ values — larger σ gives a softer, more natural-looking blur than the box filter at the same effective radius.</sub>


In [ ]:
sigmas  = [1, 3, 7]
g_imgs  = [sk_gauss(camera, sigma=s) for s in sigmas]
show_images([camera] + g_imgs,
            ["Original"] + [f"Gaussian σ={s}" for s in sigmas], figsize=(18,4))

retina_gauss = sk_gauss(retina, sigma=2)
show_images([retina, retina_gauss],
            ["Retina — original", "Retina — Gaussian σ=2"], figsize=(10,4))


### 3.3 Noise Types — Know Your Enemy

Before choosing a denoising filter it pays to identify what kind of noise you are dealing with. The four most common types are shown below.

<sub><b>Figure:</b> Common noise types: salt-and-pepper (random black/white pixels), Gaussian (continuous additive noise), Poisson (photon-count noise), and speckle (multiplicative noise).</sub>

### 3.4 Median Filter

Replaces each pixel with the **median** of its neighbourhood. Because the median is resistant to outliers, this filter excels at removing *salt-and-pepper noise* while preserving edges.

---

> **Question**: Why does the median filter preserve edges better than averaging filters?  
> At a sharp edge, half the neighbourhood is bright and half is dark. The median lands *inside* one group — so the edge stays sharp — whereas an average always falls between them, blurring the boundary.

---

<sub><b>Figure:</b> Gaussian smears the edges (see yellow inset); Median removes the speckle while keeping edges crisp.</sub>


In [ ]:
noisy = random_noise(camera, mode="s&p", amount=0.05, rng=SEED)
gauss_d  = sk_gauss(noisy, sigma=1)
median_d = ndimage.median_filter(noisy, size=3)

show_images([camera, noisy, gauss_d, median_d],
            ["Original","Salt & Pepper (5%)","Gaussian σ=1","Median 3×3"],
            figsize=(18,4))


---

## Part 4: Sharpening and Edge Detection Filters

### 4.1 Laplacian Sharpening

The Laplacian operator measures the *second derivative* of the image. Subtracting it from the original amplifies edges:

$$I_{\text{sharp}} = I - \lambda \cdot \nabla^2 I$$

<sub><b>Figure:</b> Laplacian sharpening at increasing strength λ. At λ=2.5 noise begins to be amplified alongside real edges.</sub>


In [ ]:
def sharpen_image(image, strength=1.0):
    lap = ndimage.laplace(image)
    return np.clip(image - strength * lap, 0, 1)

strengths  = [0.5, 1.0, 2.5]
sharp_imgs = [sharpen_image(camera, s) for s in strengths]
show_images([camera] + sharp_imgs,
            ["Original"] + [f"Sharpened λ={s}" for s in strengths],
            figsize=(18,4))


### 4.2 Sobel Operator — Gradient-Based Edge Detection

The Sobel operator estimates the image gradient in horizontal ($G_x$) and vertical ($G_y$) directions. The combined **gradient magnitude** $G = \sqrt{G_x^2 + G_y^2}$ highlights all edges regardless of orientation.

<sub><b>Figure:</b> Sobel decomposed into horizontal (Gx), vertical (Gy), and combined magnitude. Each catches a different subset of edges.</sub>


In [ ]:
from skimage.filters import sobel, sobel_h, sobel_v
edges_h   = sobel_h(camera)
edges_v   = sobel_v(camera)
edges_mag = sobel(camera)
show_images([camera, edges_h, edges_v, edges_mag],
            ["Original","Sobel-X (horizontal)","Sobel-Y (vertical)","Sobel magnitude"],
            figsize=(18,4))


### 4.3 Canny Edge Detector — Thin, Connected Edges

Canny works in four stages:
1. **Gaussian smoothing** — suppress noise.
2. **Gradient computation** — find magnitude and direction.
3. **Non-maximum suppression** — thin edges to one pixel wide.
4. **Hysteresis thresholding** — keep strong edges and edges connected to them.

The sample below shows how `sigma` and the two thresholds interact:

<sub><b>Figure:</b> Canny at three parameter sets. Low σ with tight thresholds gives noisy thin edges; high σ with loose thresholds gives clean, well-connected contours.</sub>

---

> **Note**: The two thresholds control a hysteresis: pixels above the *high* threshold are definitely edges; pixels between the two thresholds are kept only if they connect to a definite edge. Pixels below the *low* threshold are always discarded.

---


In [ ]:
canny_camera = canny(camera, sigma=1.5, low_threshold=0.05, high_threshold=0.15)
canny_retina = canny(retina,  sigma=2.0, low_threshold=0.02, high_threshold=0.10)

show_images([camera, canny_camera, retina, canny_retina],
            ["Camera — original","Camera — Canny edges",
             "Retina — original","Retina — Canny edges"],
            figsize=(18,4))


---

> **Practice**: Try changing the `sigma`, `low_threshold`, and `high_threshold` in the Canny call. What happens at `sigma=0.5`? What if you raise `low_threshold` to `0.2`? Make notes on how each parameter affects the result.

---


---

## Part 5: Advanced Filters

### 5.1 Bilateral Filter — Edge-Preserving Smoothing

The bilateral filter weights neighbours by *both* spatial distance **and** intensity similarity. Pixels across an edge receive near-zero weight, so the edge is preserved while smooth regions are blurred normally.

| Parameter | Controls |
|---|---|
| `d` (diameter) | How many neighbours to include |
| `sigmaColor` | Maximum intensity difference that still gets weight |
| `sigmaSpace` | Maximum spatial distance that still gets weight |

<sub><b>Figure:</b> Bilateral vs Gaussian on a noisy retina scan. Yellow insets show that bilateral preserves vessel boundaries while Gaussian blurs them.</sub>


In [ ]:
noisy_retina = random_noise(retina, mode="gaussian", var=0.005, rng=SEED)
gauss_ret  = sk_gauss(noisy_retina, sigma=2)
bilateral  = cv2.bilateralFilter(noisy_retina.astype(np.float32),
                                  d=9, sigmaColor=0.1, sigmaSpace=9)
show_images([retina, noisy_retina, gauss_ret, bilateral],
            ["Retina — original","Retina + Gaussian noise",
             "Gaussian denoised","Bilateral filter"],
            figsize=(18,4))


### 5.2 Gabor Filter — Texture and Frequency Analysis

A Gabor filter is a Gaussian-windowed sinusoid tuned to a specific *frequency* and *orientation*. Applying a bank across angles lets us build orientation-selective texture maps.

$$G(x, y; \lambda, \theta, \sigma) = \exp\!\left(-\frac{x'^2 + \gamma^2 y'^2}{2\sigma^2}\right) \cos\!\left(2\pi \frac{x'}{\lambda}\right)$$

<sub><b>Figure:</b> Gabor responses at 0°, 45°, 90°, 135° — each orientation highlights edges running in that direction. The rightmost panel is the maximum across all four.</sub>


In [ ]:
orientations = [0, np.pi/4, np.pi/2, 3*np.pi/4]
gabor_imgs, labels = [], []
for theta in orientations:
    r, im = gabor(camera, frequency=0.15, theta=theta)
    gabor_imgs.append(np.sqrt(r**2 + im**2))
    labels.append(f"Gabor θ={int(np.degrees(theta))}°")

max_resp = np.stack(gabor_imgs).max(axis=0)
show_images([camera] + gabor_imgs + [max_resp],
            ["Original"] + labels + ["Max response"],
            figsize=(22,4))


### 5.3 Non-Local Means Denoising

For each pixel, NLM searches the *entire image* for patches with similar appearance and computes a weighted average of their central pixels. This makes it extremely powerful for images with repetitive structures.

<sub><b>Figure:</b> NLM preserves fine texture details (see coat fabric) that Gaussian and Bilateral partially smooth away, at the cost of longer computation time.</sub>

---

> **Note**: NLM typically yields the highest perceptual quality but is much slower than convolution-based filters. For real-time applications, bilateral or fast Gaussian are preferred.

---


In [ ]:
noisy_cam = random_noise(camera, mode="gaussian", var=0.01, rng=SEED)
sigma_est = estimate_sigma(noisy_cam)
nlm_d = denoise_nl_means(noisy_cam, h=1.15*sigma_est, fast_mode=True,
                          patch_size=5, patch_distance=6)
show_images([camera, noisy_cam, sk_gauss(noisy_cam,sigma=1),
             cv2.bilateralFilter(noisy_cam.astype(np.float32),9,0.1,9), nlm_d],
            ["Original","Noisy","Gaussian σ=1","Bilateral","Non-Local Means"],
            figsize=(22,4))


---

## Part 6: Putting It All Together — A Practical Filtering Pipeline

We now combine several techniques into a realistic four-stage pre-processing pipeline:

| Stage | Operation | Purpose |
|---|---|---|
| ① | Bilateral denoising | Remove noise without blurring edges |
| ② | CLAHE contrast enhancement | Equalise local contrast |
| ③ | Laplacian sharpening | Restore crispness lost during denoising |
| ④ | Canny edge detection | Produce a structural map for downstream use |

<sub><b>Figure:</b> The full pipeline applied to a noisy retina scan. Each stage refines the image toward the clean ground truth on the right.</sub>


In [ ]:
def preprocess_pipeline(image):
    # Stage 1: bilateral
    denoised = cv2.bilateralFilter(image.astype(np.float32),
                                    d=9, sigmaColor=0.08, sigmaSpace=9).astype(float)
    # Stage 2: CLAHE
    clahe_op = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = img_as_float(clahe_op.apply(img_as_ubyte(np.clip(denoised,0,1))))
    # Stage 3: sharpen
    sharpened = np.clip(enhanced - 0.5*ndimage.laplace(enhanced), 0, 1)
    # Stage 4: Canny
    edges = canny(sharpened, sigma=1.5, low_threshold=0.05, high_threshold=0.15)
    return denoised, enhanced, sharpened, edges

degraded = random_noise(retina, mode="gaussian", var=0.004, rng=SEED)
d, e, s, edges = preprocess_pipeline(degraded)

show_images([degraded, d, e, s, edges, retina],
            ["① Input (noisy)","② Bilateral","③ CLAHE","④ Sharpened","⑤ Edges","Ground truth"],
            figsize=(22,4))


---

> **Practice**: Tune one parameter at a time and observe the effect:
> - Bilateral `sigmaColor` from `0.08` → `0.2` — what happens to fine vessel detail?
> - CLAHE `clipLimit` from `2.0` → `5.0` — does noise increase?
> - Sharpening `strength` from `0.5` → `2.0` — at what point is sharpening harmful?

---


---

## Part 7: Frequency-Domain Perspective (Bonus)

Every spatial filter has an equivalent operation in the **frequency domain**:

| Filter type | Frequency-domain effect |
|---|---|
| Blurring (Gaussian, box) | **Low-pass** — attenuates high frequencies |
| Sharpening | **High-pass** — amplifies high frequencies |
| Edge detection | **Band-pass / high-pass** |
| Gabor | **Narrow band-pass** at a specific frequency and orientation |

The Fourier magnitude spectra below confirm this directly:

<sub><b>Figure:</b> Top row: spatial images. Bottom row: log-scaled FFT magnitude. The blurred image has almost no energy in the outer ring (high frequencies); the sharpened image shows amplified outer-ring energy.</sub>


In [ ]:
def fft_magnitude(image):
    return np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(image))))

blurred   = sk_gauss(camera, sigma=5)
sharpened = np.clip(camera - 0.5*ndimage.laplace(camera), 0, 1)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, img, title, cmap in zip(
        axes[0], [camera, blurred, sharpened],
        ["Original","Gaussian σ=5","Sharpened"], ["gray"]*3):
    ax.imshow(img, cmap=cmap); ax.set_title(title, fontsize=12); ax.axis("off")

for ax, img, title in zip(
        axes[1], [camera, blurred, sharpened],
        ["FFT: original","FFT: blurred (low-pass)","FFT: sharpened (high-pass)"]):
    ax.imshow(fft_magnitude(img), cmap="inferno"); ax.set_title(title, fontsize=12); ax.axis("off")

plt.suptitle("Spatial images (top) and their Fourier magnitude spectra (bottom)", fontsize=13)
plt.tight_layout(); plt.show()


---

## Part 8: Filter Comparison and Selection Guide

| Filter | Best for | Preserves edges? | Speed | Main parameter(s) |
|---|---|---|---|---|
| Box / Mean | Quick preview | No | ★★★★★ | Kernel size |
| Gaussian | General smoothing | Partially | ★★★★☆ | σ |
| Median | Salt-and-pepper noise | Yes | ★★★☆☆ | Neighbourhood size |
| Bilateral | Edge-preserving denoise | Yes | ★★☆☆☆ | d, σ_color, σ_space |
| Laplacian sharpening | Detail enhancement | N/A | ★★★★☆ | λ |
| Sobel | Gradient magnitude | N/A | ★★★★★ | None |
| Canny | Thin edge maps | N/A | ★★★☆☆ | σ, lo/hi threshold |
| Gabor | Texture analysis | N/A | ★★☆☆☆ | Frequency, θ, σ |
| Non-Local Means | Best-quality denoising | Yes | ★☆☆☆☆ | h (filter strength) |

The grid below provides a direct visual and quantitative comparison of all denoising methods on the same noisy image:

<sub><b>Figure:</b> All six denoising methods on the same test image. PSNR (Peak Signal-to-Noise Ratio) is shown in each title — higher is better. NLM scores highest but is slowest.</sub>


In [ ]:
from skimage.metrics import peak_signal_noise_ratio as psnr

noisy_test = random_noise(camera, mode="gaussian", var=0.01, rng=SEED)
sigma_est  = estimate_sigma(noisy_test)

methods = {
    "Noisy input"    : noisy_test,
    "Box 5×5"        : cv2.blur(noisy_test, (5,5)),
    "Gaussian σ=2"   : sk_gauss(noisy_test, sigma=2),
    "Median 5×5"     : ndimage.median_filter(noisy_test, size=5),
    "Bilateral"      : cv2.bilateralFilter(noisy_test.astype(np.float32),9,0.1,9),
    "Non-Local Means": denoise_nl_means(noisy_test, h=1.15*sigma_est,
                           fast_mode=True, patch_size=5, patch_distance=6),
}

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, (name, img) in zip(axes.flat, methods.items()):
    clipped = np.clip(img, 0, 1)
    score   = psnr(camera, clipped)
    ax.imshow(clipped, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{name}\nPSNR = {score:.1f} dB", fontsize=11); ax.axis("off")
plt.suptitle("Denoising Filter Comparison", fontsize=14)
plt.tight_layout(); plt.show()


---

## Part 9: Conclusions and Further Reading

In this notebook we explored the theory and practice of image filtering from the humble box filter through sophisticated algorithms like non-local means and Gabor banks. Key takeaways:

- **Convolution** is the fundamental operation: a kernel slides across the image and computes a weighted sum.
- **Smoothing filters** trade off detail for noise reduction; the right choice depends on the noise model and edge-preservation requirements.
- **Sharpening filters** amplify high-frequency content — use them conservatively to avoid amplifying noise.
- **Edge detectors** (Sobel, Canny) isolate regions of rapid intensity change.
- **Gabor filters** probe specific frequencies and orientations — ideal for texture analysis.
- **Frequency-domain analysis** gives a complementary view: blurring = low-pass, sharpening = high-pass.
- In practice, **combine filters in a pipeline**: denoise → enhance contrast → detect features.

### Further reading

- *Digital Image Processing* — Gonzalez & Woods (textbook)
- [scikit-image Filters documentation](https://scikit-image.org/docs/stable/api/skimage.filters.html)
- [OpenCV Image Filtering tutorial](https://docs.opencv.org/4.x/d4/d13/tutorial_py_filtering.html)
- [Bilateral Filtering course notes — Paris, Durand, Adelson](https://people.csail.mit.edu/sparis/bf_course/)
- [Non-Local Means — Buades et al. 2005](https://www.iro.umontreal.ca/~mignotte/IFT6150/Articles/Buades-NonLocal.pdf)

---

## ***Feedback***

*Now that you have completed this chapter, reflect:*
- *Which filter surprised you most with its output?*
- *Can you think of a medical imaging problem where bilateral filtering is preferable to Gaussian blurring?*
- *Try applying the pipeline from Part 6 to an image of your own and share the result!*
